# Apply to cohort data

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.append("../..")
sys.path.append("../../evaluate_existing_models")
sys.path.append("../../develop_prediction_model/prediction_src")
sys.path.append("..")
sys.path.append("../prediction_src")
from collections import OrderedDict


from pathlib import Path
code_path = Path('/mnt/c/Users/Juliana/dev/memaef/code')
sys.path.insert(0, str(code_path))

from prediction_src.load_data import load_translation_files 
from fuzzywuzzy import process
from application_prediction import *
from collections import defaultdict
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
import pickle
from fit_prediction_model import *
from calibration import *
from evaluate_prediction_model import *
from params import *
import os

from numpy.core.multiarray import array as array
from prediction.models import LogRegModel
from prediction.coeff_func import or_to_coeff
import os
from sklearn.base import BaseEstimator, ClassifierMixin

In [5]:
dump_folder_name = "AppData_2026-02-02"
path_data = "/mnt/c/Users/Juliana/dev/memaef/data"
author = "DAG_model"

In [6]:
import copy

In [7]:
with open("../models/DAG_model_youden_durchfall.pkl","rb") as f:
    youden_model_dict = pickle.load(f)

In [8]:
sys.path.append("../../evaluate_existing_models/prediction/papermodels")

from ashrafi import AshrafiLogRegModel

In [9]:
with open("../models/DAG_model_durchfall.pkl","rb") as f:
    basis_model_dict = pickle.load(f)

In [10]:
with open("../models/DAG_model_ppv_durchfall.pkl","rb") as f:
    ppv_model_dict = pickle.load(f)

In [11]:
with open("../models/DAG_model_fpr_durchfall.pkl","rb") as f:
    fpr_model_dict = pickle.load(f)

In [13]:
questionnaire_numbers = ["2_t0","14_t0b","3_t1a", "4_t1b", ]#"6_t2b"]#,"7_t3","8_t4","13_t5"]

In [14]:
basis_model = basis_model_dict[list(basis_model_dict.keys())[0]]['model']

In [15]:
youden_model = youden_model_dict[list(youden_model_dict.keys())[0]]['model']

In [16]:
ppv_model = ppv_model_dict[list(ppv_model_dict.keys())[0]]['model']

In [17]:
fpr_model = fpr_model_dict[list(fpr_model_dict.keys())[0]]['model']

# NVF2-ELIGIBLE

In [22]:
df_nvf2_eligible = pd.read_csv("/mnt/c/Users/Juliana/dev/memaef/data/0_raw/AppData_2026-01-26/participant_2025-01-26.csv",sep=";")

df_nvf2_eligible["nvf2_eligible"] = df_nvf2_eligible["nvf2_start_date"].notna()

In [16]:
df_nvf2_eligible["nvf2_eligible"].value_counts()

nvf2_eligible
False    2910
True      556
Name: count, dtype: int64

In [84]:
df_nvf2_eligible["nvf2_status"].value_counts()

nvf2_status
CONSULT_EXPIRED      300
ACTIVE               224
CONSENT_EXPIRED       14
CONSULTATION          13
DECLINED               2
INCLUSION_EXPIRED      2
CONSENT                1
Name: count, dtype: int64

In [16]:
youden_model_dict[list(youden_model_dict.keys())[0]]["model"].threshold

0.01954109442520162

In [17]:
ppv_model_dict[list(ppv_model_dict.keys())[0]]["model"].threshold

0.156312625250501

In [18]:
fpr_model_dict[list(fpr_model_dict.keys())[0]]["model"].threshold

0.04731142174755568

In [19]:
df_t0b_ashrafi, _ = prep_model_validation(dump_folder_name = dump_folder_name,
                         path_data = "/mnt/c/Users/Juliana/dev/memaef/data",
                         questionnaire_numbers = questionnaire_numbers,
                         author = "ashrafi",
                        translate_beschwerde_cols=True)

{'diarrhea': 19.06, 'laparoscopy_and_laparotomy': 14.5, 'laparotomy': 4.1, 'pelvic_pain': 4.1, 'laparoscopy': 3.4, 'endo_family_history': 2.7, 'fatigue': 2.6, 'premenstrual_spotting': 2.2, 'galactoreahea_history': 1.8, 'dysmenorrhea': 1.8, 'dyspareunia': 1.6, 'gravidity': 0.8}


In [20]:
df_t0b_fauconnier, _ = prep_model_validation(dump_folder_name = dump_folder_name,
                         path_data = "/mnt/c/Users/Juliana/dev/memaef/data",
                         questionnaire_numbers = questionnaire_numbers,
                         author = "fauconnier",
                        translate_beschwerde_cols=True)

In [25]:
eval_var = "nvf2_eligible"

In [26]:
nvf_dict_ashrafi,nvf_metrics_ashrafi, nvf_class_metrics_ashrafi,df_application_ashrafi = return_literature_nvf_results(youden_model_dict, AshrafiLogRegModelNew(), "ashrafi",eval_var, MET_LIST,df_t0b_ashrafi, df_nvf2_eligible,"ashrafi_nvf_eligible")

{'diarrhea': 19.06, 'laparoscopy_and_laparotomy': 14.5, 'laparotomy': 4.1, 'pelvic_pain': 4.1, 'laparoscopy': 3.4, 'endo_family_history': 2.7, 'fatigue': 2.6, 'premenstrual_spotting': 2.2, 'galactoreahea_history': 1.8, 'dysmenorrhea': 1.8, 'dyspareunia': 1.6, 'gravidity': 0.8}


In [27]:
nvf_dict_fauconnier,nvf_metrics_fauconnier, nvf_class_metrics_fauconnier,df_application_fauconnier = return_literature_nvf_results(youden_model_dict, FauconnierLogRegModelNew(), "fauconnier",eval_var, MET_LIST,df_t0b_fauconnier, df_nvf2_eligible,"fauconnier_nvf_eligible")

In [88]:
nvf_metrics_fauconnier

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
fauconnier,0.59,0.14,-0.96,0.26


In [28]:
nvf_metrics_fauconnier

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
fauconnier,0.59,0.14,-0.96,0.26


In [89]:
nvf_metrics_ashrafi

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
ashrafi,0.57,0.76,-2.25,0.13


In [90]:
nvf_class_metrics_fauconnier

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.013,0.996,0.389,0.004,0.013,0.838,7,11,549,2898


In [91]:
nvf_class_metrics_ashrafi

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,1.0,0.0,0.16,1.0,1.0,0.16,556,2909,0,0


## Our model

In [102]:
def kk_model_dict_with_nvf_data_full_circle(model_dict,df_application, eval_var,new_model_dict_name, met_list):
    model = model_dict[list(model_dict.keys())[0]]["model"]
    model_dict[list(model_dict.keys())[0]]["X_test_true"] = df_application[model.coef_.index]
    model_dict[list(model_dict.keys())[0]]["y_test_true"] = df_application[eval_var]
    #model_dict = update_cutoff_dict(model_dict, new_model_dict_name)

    nvf_dict,nvf_metrics, nvf_class_metrics,df_application = return_nvf_for_eval_metrics(model_dict,df_application,eval_var,met_list)

    return nvf_dict,nvf_metrics, nvf_class_metrics,df_application

In [107]:
nvf_dict_youden,nvf_metrics_youden, nvf_class_metrics_youden,df_application_youden = kk_model_dict_with_nvf_data_full_circle(youden_model_dict,df_application, eval_var,"nvf_eligible_youden", MET_LIST)

risk_preds
1    3452
0      13
Name: count, dtype: int64


In [108]:
nvf_dict_basis,nvf_metrics_basis, nvf_class_metrics_basis,df_application_basis = kk_model_dict_with_nvf_data_full_circle(basis_model_dict,df_application, eval_var,"nvf_eligible_basis", MET_LIST)

risk_preds
0    3465
Name: count, dtype: int64


In [109]:
nvf_dict_fpr,nvf_metrics_fpr, nvf_class_metrics_fpr,df_application_fpr = kk_model_dict_with_nvf_data_full_circle(fpr_model_dict,df_application, eval_var,"nvf_eligible_fpr", MET_LIST)

risk_preds
1    3022
0     443
Name: count, dtype: int64


In [110]:
nvf_dict_ppv,nvf_metrics_ppv, nvf_class_metrics_ppv,df_application_ppv = kk_model_dict_with_nvf_data_full_circle(ppv_model_dict,df_application, eval_var,"nvf_eligible_ppv", MET_LIST)

risk_preds
0    3102
1     363
Name: count, dtype: int64


In [47]:
df_application_basis["risk_preds"].value_counts()

risk_preds
0    3465
Name: count, dtype: int64

In [48]:
df_application_youden["risk_preds"].value_counts()

risk_preds
1    3452
0      13
Name: count, dtype: int64

In [49]:
df_application_ppv["risk_preds"].value_counts()

risk_preds
0    3102
1     363
Name: count, dtype: int64

In [50]:
df_application_fpr["risk_preds"].value_counts()

risk_preds
1    3022
0     443
Name: count, dtype: int64

In [51]:
pd.crosstab(df_application["nvf2_eligible"],df_application_basis["risk_preds"])

risk_preds,0
nvf2_eligible,
False,2909
True,556


In [52]:
pd.crosstab(df_application["nvf2_eligible"],df_application_fpr["risk_preds"])

risk_preds,0,1
nvf2_eligible,,
False,393,2516
True,50,506


In [53]:
pd.crosstab(df_application["nvf2_eligible"],df_application_ppv["risk_preds"])

risk_preds,0,1
nvf2_eligible,,
False,2618,291
True,484,72


In [105]:
pd.crosstab(df_application["nvf2_eligible"],df_application_youden["risk_preds"])

risk_preds,0,1
nvf2_eligible,,
False,13,2896
True,0,556


In [55]:
classification_metrics(basis_model_dict,basis_model.threshold).round(3)

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.0,1.0,NaN,0.0,0.0,0.84,0,0,556,2909


In [56]:
classification_metrics(youden_model_dict,youden_model.threshold).round(3)

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,1.0,0.004,0.161,0.996,1.0,0.164,556,2896,0,13


In [106]:
classification_metrics(nvf_dict_youden,nvf_dict_youden[list(nvf_dict_youden.keys())[0]]["model"].threshold).round(3)

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,1.0,0.004,0.161,0.996,1.0,0.164,556,2896,0,13


In [57]:
classification_metrics(ppv_model_dict,ppv_model.threshold).round(3)

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.129,0.9,0.198,0.1,0.129,0.776,72,291,484,2618


In [58]:
classification_metrics(fpr_model_dict,fpr_model.threshold).round(3)

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.91,0.135,0.167,0.865,0.91,0.259,506,2516,50,393


In [41]:
nvf_metrics_basis.round(3)

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.54,0.14,-1.11,0.24


In [64]:
nvf_class_metrics_ppv

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.129,0.9,0.198,0.1,0.129,0.776,72,291,484,2618


In [65]:
nvf_class_metrics_fpr

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.91,0.135,0.167,0.865,0.91,0.259,506,2516,50,393


In [66]:
nvf_class_metrics_youden

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,1.0,0.004,0.161,0.996,1.0,0.164,556,2896,0,13


In [67]:
nvf_class_metrics_basis

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.0,1.0,NaN,0.0,0.0,0.84,0,0,556,2909


In [26]:
df_application[youden_model.coef_.index].apply(lambda x:x.value_counts())

,Verdauungsstörungen,Blutung,Dyspareunie,Menstruationsschmerzen,Bauchschmerzen,Urinieren,Verstopfung
False,2523,2747,2214,68,589,3124,3465.0
True,942,718,1251,3397,2876,341,NaN


In [27]:
df_application["risk_preds"].value_counts()

risk_preds
1    3452
0      13
Name: count, dtype: int64

In [28]:
df_application[df_application["risk_preds"]==1]["risk_probs_1"].mean().round(3)

0.104

In [29]:
df_application[df_application["risk_preds"]==1]["risk_probs_1"].std().round(3)

0.061

In [30]:
df_application[df_application["risk_preds"]==1]["risk_probs_1"].min().round(3),df_application[df_application["risk_preds"]==1]["risk_probs_1"].max().round(3)

(0.021, 0.371)

In [31]:
np.quantile(df_application[df_application["risk_preds"]==1]["risk_probs_1"].values,[0.25,0.75]).round(3)

array([0.072, 0.111])

----------------

In [32]:
pd.crosstab(df_application["Bauchschmerzen"],df_application["risk_preds"])

risk_preds,0,1
Bauchschmerzen,,
False,13,576
True,0,2876


-------------------

In [33]:
pd.crosstab(df_application["Menstruationsschmerzen"],df_application["risk_preds"])

risk_preds,0,1
Menstruationsschmerzen,,
False,13,55
True,0,3397


In [34]:
pd.crosstab(df_application["nvf2_eligible"],df_application["risk_preds"])

risk_preds,0,1
nvf2_eligible,,
False,13,2896
True,0,556


In [35]:
df_application[df_application["risk_preds"]==0][youden_model.coef_.index].apply(lambda x:x.value_counts())

,Verdauungsstörungen,Blutung,Dyspareunie,Menstruationsschmerzen,Bauchschmerzen,Urinieren,Verstopfung
False,13,13,13,13,13,13,13


In [36]:
df_application[df_application["nvf2_eligible"]==0][youden_model.coef_.index].apply(lambda x:x.value_counts())

,Verdauungsstörungen,Blutung,Dyspareunie,Menstruationsschmerzen,Bauchschmerzen,Urinieren,Verstopfung
False,2164,2309,1848,61,508,2635,2909.0
True,745,600,1061,2848,2401,274,NaN


In [37]:
df_application[df_application["risk_preds"]==1][youden_model.coef_.index].apply(lambda x:x.value_counts())

,Verdauungsstörungen,Blutung,Dyspareunie,Menstruationsschmerzen,Bauchschmerzen,Urinieren,Verstopfung
False,2510,2734,2201,55,576,3111,3452.0
True,942,718,1251,3397,2876,341,NaN


In [38]:
df_application[df_application["nvf2_eligible"]==1][youden_model.coef_.index].apply(lambda x:x.value_counts())

,Verdauungsstörungen,Blutung,Dyspareunie,Menstruationsschmerzen,Bauchschmerzen,Urinieren,Verstopfung
False,359,438,366,7,81,489,556.0
True,197,118,190,549,475,67,NaN


In [40]:
youden_model.coef_table_.round(3)

,feature,coef,odds_ratio,p_value,ci_lower,ci_upper
0,Intercept,-4.182,0.015,0.000,-4.251,-4.114
1,Verdauungsstörungen,0.378,1.460,0.041,0.016,0.741
2,Blutung,0.476,1.610,0.000,0.249,0.703
3,Dyspareunie,0.361,1.435,0.625,-1.087,1.809
4,Menstruationsschmerzen,0.809,2.246,0.000,0.681,0.937
5,Bauchschmerzen,0.818,2.267,0.000,0.688,0.949
6,Urinieren,0.813,2.255,0.000,0.465,1.161
7,Verstopfung,-0.112,0.894,0.829,-1.125,0.902


-------------------

# ENDOMETRIOSE

In [68]:
ll = os.listdir("/mnt/c/Users/Juliana/dev/memaef/data/1_processed_data/AppData_2026-02-02")

In [29]:
# these are the questionnaires we choose to get the Selbstauskunft-Endometriosediagnose from (T2 until end)
ll= ['10_t6.csv',
 '11_t7.csv',
 '12_t8.csv',
 '13_t5.csv',
 '18_r0a.csv',
 '23_t2a.csv',
 '26_r0b.csv',
 '27_r1ig.csv',
 '28_r1wg.csv',
 '29_r2ig.csv',
 '30_r2wg.csv',
 '31_r3ig.csv',
 '32_r3wg.csv',
 '35_r5ig.csv',
 '36_r5wg.csv',
 '38_r6wg.csv',
 '47_r4ig.csv',
 '48_r4wg.csv',
 '6_t2b.csv',
 '7_t3.csv',
 '8_t4.csv']

In [30]:
ll = [l.split(".csv")[0] for l in ll]

In [31]:
df_t0b,_ = load_prediction_tables(experiment_name=dump_folder_name,
                           path_data="/mnt/c/Users/Juliana/dev/memaef/data",
                           questionnaire_numbers=ll)

df_diagnosis = prep_nvf_diagnosis_df(df_t0b)

df_diagnosis= pd.merge(df_diagnosis,df_nvf2_eligible[["pseudonym","nvf2_eligible"]],on="pseudonym",how="right")

In [98]:
df_diagnosis["pseudonym"].nunique()

3465

In [101]:
df_diagnosis["diagnose"].fillna(0,inplace=True)

In [102]:
df_diagnosis["diagnose"].value_counts()

diagnose
0.0    3416
1.0      50
Name: count, dtype: int64

In [103]:
pd.crosstab(df_diagnosis["nvf2_eligible"],df_diagnosis["diagnose"])

diagnose,0.0,1.0
nvf2_eligible,,
False,2897,13
True,519,37


In [51]:
df_diagnosis["diagnose"].value_counts()

diagnose
0    1027
1      50
Name: count, dtype: int64

In [32]:
eval_var = "diagnose"

In [34]:
nvf_dict_ashrafi2,nvf_metrics_ashrafi2, nvf_class_metrics_ashrafi2,df_application_ashrafi2 = return_literature_nvf_results(youden_model_dict, AshrafiLogRegModelNew(), "ashrafi",eval_var, MET_LIST,df_t0b_ashrafi, df_diagnosis,"ashrafi_diagnosis")

{'diarrhea': 19.06, 'laparoscopy_and_laparotomy': 14.5, 'laparotomy': 4.1, 'pelvic_pain': 4.1, 'laparoscopy': 3.4, 'endo_family_history': 2.7, 'fatigue': 2.6, 'premenstrual_spotting': 2.2, 'galactoreahea_history': 1.8, 'dysmenorrhea': 1.8, 'dyspareunia': 1.6, 'gravidity': 0.8}


In [35]:
nvf_dict_fauconnier2,nvf_metrics_fauconnier2, nvf_class_metrics_fauconnier2,df_application_fauconnier2 = return_literature_nvf_results(youden_model_dict, FauconnierLogRegModelNew(), "fauconnier",eval_var, MET_LIST,df_t0b_fauconnier, df_diagnosis,"fauconnier_diagnosis")

In [40]:
nvf_metrics_ashrafi

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
ashrafi,0.57,0.76,-2.25,0.13


In [41]:
nvf_class_metrics_ashrafi

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,1.0,0.0,0.16,1.0,1.0,0.16,556,2909,0,0


In [36]:
nvf_metrics_ashrafi2

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
ashrafi,0.6,0.89,-4.5,0.08


In [37]:
nvf_class_metrics_ashrafi2

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,1.0,0.0,0.014,1.0,1.0,0.014,50,3415,0,0


In [42]:
nvf_metrics_fauconnier

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
fauconnier,0.59,0.14,-0.96,0.26


In [43]:
nvf_class_metrics_fauconnier

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.013,0.996,0.389,0.004,0.013,0.838,7,11,549,2898


In [38]:
nvf_metrics_fauconnier2

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
fauconnier,0.6,0.03,-2.97,0.48


In [39]:
nvf_class_metrics_fauconnier2

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.02,0.995,0.056,0.005,0.02,0.981,1,17,49,3398


## With our model:

In [116]:
df_application2 = prepare_dag_model_df(df_t0b_author, df_diagnosis,eval_var,youden_model)

In [117]:
nvf_dict_youden2,nvf_metrics_youden2, nvf_class_metrics_youden2,df_application_youden2 = kk_model_dict_with_nvf_data_full_circle(youden_model_dict,df_application2, eval_var,"nvf_diagnose_youden", MET_LIST)

risk_preds
1    3452
0      13
Name: count, dtype: int64


In [118]:
nvf_dict_basis2,nvf_metrics_basis2, nvf_class_metrics_basis2,df_application_basis2 = kk_model_dict_with_nvf_data_full_circle(basis_model_dict,df_application2, eval_var,"nvf_diagnose_basis", MET_LIST)

risk_preds
0    3465
Name: count, dtype: int64


In [119]:
nvf_dict_fpr2,nvf_metrics_fpr2, nvf_class_metrics_fpr2,df_application_fpr2 = kk_model_dict_with_nvf_data_full_circle(fpr_model_dict,df_application2, eval_var,"nvf_diagnose_fpr", MET_LIST)

risk_preds
1    3022
0     443
Name: count, dtype: int64


In [120]:
nvf_dict_ppv2,nvf_metrics_ppv2, nvf_class_metrics_ppv2,df_application_ppv2 = kk_model_dict_with_nvf_data_full_circle(ppv_model_dict,df_application2, eval_var,"nvf_diagnose_ppv", MET_LIST)

risk_preds
0    3102
1     363
Name: count, dtype: int64


In [121]:
df_application_basis2["risk_preds"].value_counts()

risk_preds
0    3465
Name: count, dtype: int64

In [122]:
df_application_youden2["risk_preds"].value_counts()

risk_preds
1    3452
0      13
Name: count, dtype: int64

In [123]:
df_application_ppv2["risk_preds"].value_counts()

risk_preds
0    3102
1     363
Name: count, dtype: int64

In [124]:
df_application_fpr2["risk_preds"].value_counts()

risk_preds
1    3022
0     443
Name: count, dtype: int64

In [125]:
pd.crosstab(df_application_basis2[eval_var],df_application_basis2["risk_preds"])

risk_preds,0
diagnose,
0.0,3415
1.0,50


In [126]:
pd.crosstab(df_application_youden2[eval_var],df_application_youden2["risk_preds"])

risk_preds,0,1
diagnose,,
0.0,13,3402
1.0,0,50


In [127]:
pd.crosstab(df_application_fpr2[eval_var],df_application_fpr2["risk_preds"])

risk_preds,0,1
diagnose,,
0.0,440,2975
1.0,3,47


In [128]:
pd.crosstab(df_application_ppv2[eval_var],df_application_ppv2["risk_preds"])

risk_preds,0,1
diagnose,,
0.0,3063,352
1.0,39,11


In [130]:
pd.crosstab(df_application_ppv["nvf2_eligible"],df_application_ppv["risk_preds"])

risk_preds,0,1
nvf2_eligible,,
False,2618,291
True,484,72


In [131]:
nvf_metrics_basis2.round(3)

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.59,0.03,-2.5,0.78


In [132]:
nvf_metrics_basis.round(3)

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.54,0.14,-1.11,0.24


In [133]:
nvf_class_metrics_ppv2

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.22,0.897,0.03,0.103,0.22,0.887,11,352,39,3063


In [134]:
nvf_class_metrics_fpr2

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.94,0.129,0.016,0.871,0.94,0.141,47,2975,3,440


In [135]:
nvf_class_metrics_youden2

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,1.0,0.004,0.014,0.996,1.0,0.018,50,3402,0,13


In [136]:
nvf_class_metrics_basis2

,Sensitivity,Specificity,PPV,FPR,TPR,Accuracy,TP,FP,FN,TN
0,0.0,1.0,NaN,0.0,0.0,0.986,0,0,50,3415


In [137]:
nvf_metrics_basis2.round(3)

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.59,0.03,-2.5,0.78


In [138]:
nvf_metrics_youden2.round(3)

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.59,0.03,-2.5,0.78


In [139]:
nvf_metrics_ppv2.round(3)

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.59,0.03,-2.5,0.78


In [81]:
nvf_metricsdurch_fpr2.round(3)

,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
Model,,,,
endo_data_as_groupedTrue_is_sumFalse_nvfFalse,0.59,0.03,-2.5,0.78


In [140]:
df_application2[youden_model.coef_.index].apply(lambda x:x.value_counts())

,Menstruationsschmerzen,Bauchschmerzen,Verdauungsstörungen,Verstopfung,Blutung,Urinieren,Dyspareunie
False,68,589,2523,3465.0,2747,3124,2214
True,3397,2876,942,NaN,718,341,1251


In [143]:
pd.crosstab(df_application_youden2["Bauchschmerzen"],df_application_youden2["risk_preds"])

risk_preds,0,1
Bauchschmerzen,,
False,13,576
True,0,2876


-------------------

In [144]:
pd.crosstab(df_application_youden2["Menstruationsschmerzen"],df_application_youden2["risk_preds"])

risk_preds,0,1
Menstruationsschmerzen,,
False,13,55
True,0,3397


In [145]:
pd.crosstab(df_application_youden2["diagnose"],df_application_youden2["risk_preds"])

risk_preds,0,1
diagnose,,
0.0,13,3402
1.0,0,50


In [147]:
pd.crosstab(df_application_youden2[eval_var],df_application_youden2["Menstruationsschmerzen"])

Menstruationsschmerzen,False,True
diagnose,,
0.0,68,3347
1.0,0,50


In [146]:
pd.crosstab(df_application_youden2[eval_var],df_application_youden2["Bauchschmerzen"])

Bauchschmerzen,False,True
diagnose,,
0.0,580,2835
1.0,9,41


In [70]:
df_application_youden2[df_application_youden2["risk_preds"]==0][youden_model.coef_.index].apply(lambda x:x.value_counts())

,Verdauungsstörungen,Blutung,Dyspareunie,Menstruationsschmerzen,Bauchschmerzen,Urinieren,Verstopfung
False,13,13,13,13,13,13,13


In [71]:
df_application_youden2[df_application_youden2[eval_var]==0][youden_model.coef_.index].apply(lambda x:x.value_counts())

,Verdauungsstörungen,Blutung,Dyspareunie,Menstruationsschmerzen,Bauchschmerzen,Urinieren,Verstopfung
False,2493,2712,2184,68,580,3085,3415.0
True,922,703,1231,3347,2835,330,NaN


In [72]:
df_application_youden2[df_application_youden2["risk_preds"]==1][youden_model.coef_.index].apply(lambda x:x.value_counts())

,Verdauungsstörungen,Blutung,Dyspareunie,Menstruationsschmerzen,Bauchschmerzen,Urinieren,Verstopfung
False,2510,2734,2201,55,576,3111,3452.0
True,942,718,1251,3397,2876,341,NaN


In [73]:
df_application_youden2[df_application_youden2[eval_var]==1][youden_model.coef_.index].apply(lambda x:x.value_counts())

,Verdauungsstörungen,Blutung,Dyspareunie,Menstruationsschmerzen,Bauchschmerzen,Urinieren,Verstopfung
False,30,35,30,NaN,9,39,50.0
True,20,15,20,50.0,41,11,NaN


In [74]:
youden_model.coef_.round(3)

Verdauungsstörungen       0.378
Blutung                   0.476
Dyspareunie               0.361
Menstruationsschmerzen    0.809
Bauchschmerzen            0.818
Urinieren                 0.813
Verstopfung              -0.112
dtype: float64

-------------------